# DMET-SSTKG — Source S-STKG Phase Classifier

This notebook reproduces the source phase-classification stage from the
released role-normalized S-STKG data.

Before running the notebook, the user edits only two paths in the
configuration cell:

- `BASE_PATH`: the existing 20-case source S-STKG directory;
- `RESULTS_ROOT`: a writable directory for generated outputs.

No prebuilt binary cache is required. The notebook automatically creates
`RESULTS_ROOT/source_phase_classifier_binary_cache/` on the first run.
A complete compatible cache is reused on later runs; an incomplete or
incompatible cache is rebuilt automatically. The user does not need to provide
a cache path or change any cache-control flag.

The notebook:

- validates that all 20 source-development cases are present;
- prevents source data from overlapping generated output directories;
- builds the cache in a temporary directory and publishes it only after
  successful completion;
- identifies sampled frames with the `(g_ratio, p_ratio)` pair;
- searches 10,000 candidate case-disjoint 14/3/3 splits; and
- writes a timestamped run containing the checkpoint, split artifacts,
  reports, and figures.

Expected source layout:

```text
<BASE_PATH>/
    Case_01/
        <fragment>_nodes.txt
        <fragment>_edges.txt
        <fragment>_edge_features.txt
        <fragment>_labels.txt
    ...
    Case_20/
```


## Environment

The tested repository environment uses Python 3.9, PyTorch 2.4.1,
PyTorch Geometric, pandas, NumPy, Matplotlib, scikit-learn, tqdm, and
Jupyter.

```bash
python -m pip install torch==2.4.1
python -m pip install torch-geometric pandas numpy matplotlib scikit-learn tqdm jupyter
```

Absolute paths are recommended in the configuration cell so that notebook
behavior does not depend on the directory from which Jupyter was launched.


In [ ]:
import os
import json
import shutil
import time
import hashlib
import random
from datetime import datetime
from pathlib import Path

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GATv2Conv
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)

In [ ]:
# =========================================================
# 0) CONFIG
# =========================================================

# Existing source S-STKG directory containing Case_01, ..., Case_20.
# Replace only this example path with the local source-data directory.
BASE_PATH = Path(
    "/path/to/source_sstkg_directory"
)

# Writable directory for the generated cache, checkpoints, split files,
# reports, and figures. Replace only this example path.
RESULTS_ROOT = Path(
    "/path/to/output_directory"
)

# No prebuilt cache is required and the user does not edit this path.
# The notebook creates the cache automatically on the first run and
# reuses a complete compatible cache on subsequent runs.
BIN_PATH = (
    RESULTS_ROOT
    / "source_phase_classifier_binary_cache"
)

SEED = 42

CHUNK_SIZE = 100
MIN_CHUNK_FRAMES = 10
NUM_PHASES = 8
HIDDEN_DIM = 128
NUM_EPOCHS = 40
LR = 0.001
WEIGHT_DECAY = 0.01
GRAD_CLIP_NORM = 1.0

# =========================================================
# FEATURE CLEANING
# =========================================================
# Original node feature layout:
# x,y,w,h,vx,vy,class,case_id,g_ratio,p_ratio,isV
#
# case_id is zeroed because it is a case-specific shortcut. g_ratio and
# p_ratio remain available as temporal-position features.
REMOVE_CASE_ID_FEATURE = True
REMOVE_G_RATIO_FEATURE = False
REMOVE_P_RATIO_FEATURE = False

CASE_ID_COL = 7
G_RATIO_COL = 8
P_RATIO_COL = 9
VIRTUAL_NODE_COL = 10

TRAIN_CASES_N = 14
VAL_CASES_N = 3
TEST_CASES_N = 3
EXPECTED_NUM_CASES = 20
CASE_SPLIT_SEARCH_TRIALS = 10000

CACHE_SCHEMA_VERSION = 2
FRAME_KEY_TOLERANCE = 1e-6

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PHASE_NAMES = [
    "TrocarPlacement",
    "Preparation",
    "CalotTriangleDissection",
    "ClippingCutting",
    "GallbladderDissection",
    "GallbladderPackaging",
    "CleaningCoagulation",
    "GallbladderRetraction",
]


def paths_overlap(path_a, path_b):
    """True when paths are equal or one is an ancestor of the other."""
    path_a = Path(path_a).expanduser().resolve()
    path_b = Path(path_b).expanduser().resolve()

    return (
        path_a == path_b
        or path_a in path_b.parents
        or path_b in path_a.parents
    )


def discover_case_directories(root_dir):
    root_dir = Path(root_dir).expanduser().resolve()

    if not root_dir.is_dir():
        return []

    return sorted(
        path
        for path in root_dir.iterdir()
        if (
            path.is_dir()
            and path.name.startswith("Case_")
        )
    )


def validate_repository_paths():
    source_root = Path(
        BASE_PATH
    ).expanduser().resolve()

    cache_root = Path(
        BIN_PATH
    ).expanduser().resolve()

    results_root = Path(
        RESULTS_ROOT
    ).expanduser().resolve()

    if not source_root.is_dir():
        raise FileNotFoundError(
            "Source S-STKG directory was not found:\n"
            f"{source_root}\n\n"
            "Update BASE_PATH in the configuration cell."
        )

    if paths_overlap(
        source_root,
        results_root,
    ):
        raise RuntimeError(
            "RESULTS_ROOT must be separate from BASE_PATH "
            "and must not contain the source data.\n"
            f"BASE_PATH   : {source_root}\n"
            f"RESULTS_ROOT: {results_root}"
        )

    if paths_overlap(
        source_root,
        cache_root,
    ):
        raise RuntimeError(
            "The automatically generated binary cache "
            "must be separate from BASE_PATH.\n"
            f"BASE_PATH: {source_root}\n"
            f"BIN_PATH : {cache_root}"
        )

    case_directories = discover_case_directories(
        source_root
    )

    if len(case_directories) != EXPECTED_NUM_CASES:
        raise RuntimeError(
            f"Expected exactly {EXPECTED_NUM_CASES} source "
            f"Case_* folders but found "
            f"{len(case_directories)} under:\n"
            f"{source_root}\n\n"
            "Detected cases: "
            f"{[path.name for path in case_directories]}"
        )

    if (
        TRAIN_CASES_N
        + VAL_CASES_N
        + TEST_CASES_N
        != EXPECTED_NUM_CASES
    ):
        raise RuntimeError(
            "Train/validation/test case counts do not "
            f"sum to {EXPECTED_NUM_CASES}."
        )

    results_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(f"[OK] Source S-STKG : {source_root}")
    print(f"[OK] Source cases  : {len(case_directories)}")
    print(f"[OK] Binary cache  : {cache_root}")
    print(f"[OK] Results root  : {results_root}")
    print(f"[OK] Device        : {DEVICE}")

    return case_directories


SOURCE_CASE_DIRECTORIES = validate_repository_paths()


In [ ]:
# =========================================================
# 1) REPRODUCIBILITY
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# =========================================================
# 2) LABEL HELPER
# =========================================================
def safe_label_to_zero_based(raw_label, num_classes=8):
    raw_label = int(raw_label)

    if 1 <= raw_label <= num_classes:
        return raw_label - 1

    if 0 <= raw_label < num_classes:
        return raw_label

    raise ValueError(f"Unexpected phase label value: {raw_label}")

In [ ]:
# =========================================================
# 3) DATASET
# =========================================================
class SurgicalTurboDataset(Dataset):
    """Load S-STKG fragments through a complete, self-validating cache."""

    REQUIRED_SUFFIXES = (
        "_nodes.txt",
        "_edges.txt",
        "_edge_features.txt",
        "_labels.txt",
    )

    def __init__(
        self,
        root_dir,
        binary_dir,
        chunk_size=100,
        minimum_chunk_frames=10,
    ):
        self.root_dir = Path(root_dir).expanduser().resolve()
        self.binary_dir = Path(binary_dir).expanduser().resolve()
        self.chunk_size = int(chunk_size)
        self.minimum_chunk_frames = int(minimum_chunk_frames)

        if paths_overlap(self.root_dir, self.binary_dir):
            raise RuntimeError(
                "Refusing to use a binary cache that overlaps the source "
                f"data.\nSource: {self.root_dir}\nCache : {self.binary_dir}"
            )

        super().__init__(str(self.root_dir))

        self.case_directories = discover_case_directories(self.root_dir)
        if len(self.case_directories) != EXPECTED_NUM_CASES:
            raise RuntimeError(
                f"Expected {EXPECTED_NUM_CASES} source cases but found "
                f"{len(self.case_directories)}."
            )

        self.source_fingerprint = self._compute_source_fingerprint()
        cache_is_valid, reason = self._cache_is_valid()

        if cache_is_valid:
            print(f"[OK] Complete binary cache is reused: {self.binary_dir}")
        else:
            print(f"[INFO] Binary cache will be rebuilt: {reason}")
            self._convert_and_fragment()

        self.files = sorted(self.binary_dir.glob("frag_*.pt"))
        if not self.files:
            raise RuntimeError(f"Binary cache is empty: {self.binary_dir}")

    @property
    def manifest_path(self):
        return self.binary_dir / "cache_manifest.json"

    @property
    def expected_case_ids(self):
        return [path.name for path in self.case_directories]

    def _source_files(self):
        source_files = []
        for case_directory in self.case_directories:
            for source_file in case_directory.iterdir():
                if (
                    source_file.is_file()
                    and source_file.name.endswith(self.REQUIRED_SUFFIXES)
                ):
                    source_files.append(source_file)
        return sorted(source_files)

    def _compute_source_fingerprint(self):
        source_files = self._source_files()
        if not source_files:
            raise RuntimeError(
                f"No S-STKG text files were found under {self.root_dir}."
            )

        hasher = hashlib.sha256()
        for source_file in source_files:
            stat = source_file.stat()
            relative_path = source_file.relative_to(self.root_dir)
            record = (
                f"{relative_path.as_posix()}|{stat.st_size}|"
                f"{stat.st_mtime_ns}\n"
            )
            hasher.update(record.encode("utf-8"))
        return hasher.hexdigest()

    def _expected_manifest_fields(self):
        return {
            "cache_schema_version": CACHE_SCHEMA_VERSION,
            "source_root": str(self.root_dir),
            "source_fingerprint": self.source_fingerprint,
            "case_ids": self.expected_case_ids,
            "chunk_size": self.chunk_size,
            "minimum_chunk_frames": self.minimum_chunk_frames,
            "frame_key": ["g_ratio", "p_ratio"],
            "feature_cleaning": {
                "case_id_zeroed": bool(REMOVE_CASE_ID_FEATURE),
                "g_ratio_zeroed": bool(REMOVE_G_RATIO_FEATURE),
                "p_ratio_zeroed": bool(REMOVE_P_RATIO_FEATURE),
            },
        }

    def _cache_is_valid(self):
        if not self.binary_dir.is_dir():
            return False, "cache directory does not exist"
        if not self.manifest_path.is_file():
            return False, "cache completion manifest is missing"

        try:
            manifest = json.loads(
                self.manifest_path.read_text(encoding="utf-8")
            )
        except Exception as error:
            return False, f"cache manifest cannot be read: {error}"

        for key, expected_value in self._expected_manifest_fields().items():
            if manifest.get(key) != expected_value:
                return False, (
                    f"manifest field {key!r} does not match the current "
                    "input or configuration"
                )

        fragment_files = sorted(self.binary_dir.glob("frag_*.pt"))
        expected_fragments = int(manifest.get("num_fragments", -1))

        if expected_fragments <= 0:
            return False, "manifest reports no completed fragments"
        if len(fragment_files) != expected_fragments:
            return False, (
                "cache is incomplete: manifest reports "
                f"{expected_fragments} fragments but found "
                f"{len(fragment_files)}"
            )

        return True, "cache is complete"

    @staticmethod
    def _safe_remove_directory(directory, source_root):
        directory = Path(directory).expanduser().resolve()
        source_root = Path(source_root).expanduser().resolve()

        if (
            directory in {Path('/').resolve(), Path.home().resolve()}
            or paths_overlap(directory, source_root)
        ):
            raise RuntimeError(
                f"Unsafe directory-removal request rejected: {directory}"
            )

        if directory.exists():
            shutil.rmtree(directory)

    @staticmethod
    def _pair_mask(table, g_ratio, p_ratio):
        g_mask = np.isclose(
            table.iloc[:, 0].to_numpy(dtype=float),
            float(g_ratio),
            rtol=0.0,
            atol=FRAME_KEY_TOLERANCE,
        )
        p_mask = np.isclose(
            table.iloc[:, 1].to_numpy(dtype=float),
            float(p_ratio),
            rtol=0.0,
            atol=FRAME_KEY_TOLERANCE,
        )
        return g_mask & p_mask

    @staticmethod
    def _validate_tables(n_df, e_df, ef_df, label_df, fragment_prefix):
        if n_df.shape[1] < 11:
            raise RuntimeError(
                f"{fragment_prefix}: node table has {n_df.shape[1]} "
                "columns; expected at least 11."
            )
        if e_df.shape[1] < 5:
            raise RuntimeError(
                f"{fragment_prefix}: edge table has {e_df.shape[1]} "
                "columns; expected at least 5."
            )
        if ef_df.shape[1] < 7:
            raise RuntimeError(
                f"{fragment_prefix}: edge-feature table has "
                f"{ef_df.shape[1]} columns; expected at least 7."
            )
        if label_df.empty or label_df.shape[1] < 2:
            raise RuntimeError(
                f"{fragment_prefix}: label table is empty or invalid."
            )

    def _convert_and_fragment(self):
        print("\n[INFO] Converting role-normalized S-STKGs into binary fragments...")

        temporary_dir = (
            self.binary_dir.parent / f".{self.binary_dir.name}.building"
        )
        self._safe_remove_directory(temporary_dir, self.root_dir)
        temporary_dir.mkdir(parents=True, exist_ok=False)

        global_idx = 0
        case_fragment_counts = {
            case_id: 0 for case_id in self.expected_case_ids
        }

        try:
            for case_directory in tqdm(
                self.case_directories,
                desc="Building source binary cache",
            ):
                prefixes = sorted(
                    {
                        path.name.removesuffix("_nodes.txt")
                        for path in case_directory.glob("*_nodes.txt")
                    }
                )

                if not prefixes:
                    raise RuntimeError(
                        f"No *_nodes.txt files found in {case_directory}."
                    )

                for pref in prefixes:
                    paths = {
                        "nodes": case_directory / f"{pref}_nodes.txt",
                        "edges": case_directory / f"{pref}_edges.txt",
                        "edge_features": (
                            case_directory / f"{pref}_edge_features.txt"
                        ),
                        "labels": case_directory / f"{pref}_labels.txt",
                    }

                    missing = [
                        str(path) for path in paths.values() if not path.is_file()
                    ]
                    if missing:
                        raise FileNotFoundError(
                            f"Incomplete S-STKG fragment {pref}; missing:\n"
                            + "\n".join(missing)
                        )

                    n_df = pd.read_csv(paths["nodes"])
                    e_df = pd.read_csv(paths["edges"])
                    ef_df = pd.read_csv(paths["edge_features"])
                    label_df = pd.read_csv(paths["labels"])

                    self._validate_tables(n_df, e_df, ef_df, label_df, pref)

                    raw_label = int(label_df.iloc[0, 1])
                    label = safe_label_to_zero_based(raw_label, NUM_PHASES)

                    frame_keys = np.unique(
                        n_df.iloc[:, [G_RATIO_COL, P_RATIO_COL]].to_numpy(
                            dtype=float
                        ),
                        axis=0,
                    )
                    if frame_keys.size == 0:
                        continue

                    frame_keys = frame_keys[
                        np.lexsort((frame_keys[:, 1], frame_keys[:, 0]))
                    ]

                    for chunk_start in range(0, len(frame_keys), self.chunk_size):
                        chunk_frame_keys = frame_keys[
                            chunk_start:chunk_start + self.chunk_size
                        ]

                        if len(chunk_frame_keys) < self.minimum_chunk_frames:
                            continue

                        graph_seq = []

                        for frame_g_ratio, frame_p_ratio in chunk_frame_keys:
                            node_mask = (
                                np.isclose(
                                    n_df.iloc[:, G_RATIO_COL].to_numpy(dtype=float),
                                    float(frame_g_ratio),
                                    rtol=0.0,
                                    atol=FRAME_KEY_TOLERANCE,
                                )
                                & np.isclose(
                                    n_df.iloc[:, P_RATIO_COL].to_numpy(dtype=float),
                                    float(frame_p_ratio),
                                    rtol=0.0,
                                    atol=FRAME_KEY_TOLERANCE,
                                )
                            )
                            f_n = n_df.loc[node_mask].to_numpy(
                                dtype=np.float32,
                                copy=True,
                            )

                            if len(f_n) == 0:
                                raise RuntimeError(
                                    f"Empty frame in {pref}: "
                                    f"g_ratio={frame_g_ratio}, "
                                    f"p_ratio={frame_p_ratio}"
                                )

                            if REMOVE_CASE_ID_FEATURE:
                                f_n[:, CASE_ID_COL] = 0.0
                            if REMOVE_G_RATIO_FEATURE:
                                f_n[:, G_RATIO_COL] = 0.0
                            if REMOVE_P_RATIO_FEATURE:
                                f_n[:, P_RATIO_COL] = 0.0

                            edge_mask = self._pair_mask(
                                e_df, frame_g_ratio, frame_p_ratio
                            )
                            edge_feature_mask = self._pair_mask(
                                ef_df, frame_g_ratio, frame_p_ratio
                            )
                            f_e = e_df.loc[edge_mask].to_numpy(copy=True)
                            f_ef = ef_df.loc[edge_feature_mask].to_numpy(copy=True)

                            if len(f_e) != len(f_ef):
                                raise RuntimeError(
                                    "Edge and edge-feature row counts differ for "
                                    f"{pref}, g_ratio={frame_g_ratio}, "
                                    f"p_ratio={frame_p_ratio}: "
                                    f"{len(f_e)} versus {len(f_ef)}"
                                )

                            if len(f_e) > 0:
                                e_idx = torch.tensor(
                                    f_e[:, 2:4].astype(np.int64),
                                    dtype=torch.long,
                                ).t().contiguous()

                                if (
                                    int(e_idx.min()) < 0
                                    or int(e_idx.max()) >= len(f_n)
                                ):
                                    raise RuntimeError(
                                        f"Frame-local edge index is invalid in {pref}."
                                    )
                            else:
                                e_idx = torch.zeros((2, 0), dtype=torch.long)

                            if len(f_ef) > 0:
                                e_at = torch.tensor(
                                    f_ef[:, 4:7].astype(np.float32),
                                    dtype=torch.float,
                                )
                            else:
                                e_at = torch.zeros((0, 3), dtype=torch.float)

                            data = Data(
                                x=torch.tensor(f_n, dtype=torch.float),
                                edge_index=e_idx,
                                edge_attr=e_at,
                                y=torch.tensor([label], dtype=torch.long),
                            )
                            data.is_v = data.x[:, VIRTUAL_NODE_COL] == 1

                            virtual_count = int(data.is_v.sum().item())
                            if virtual_count != 1:
                                raise RuntimeError(
                                    "Expected exactly one virtual node per sampled "
                                    f"frame in {pref}; found {virtual_count} at "
                                    f"g_ratio={frame_g_ratio}, "
                                    f"p_ratio={frame_p_ratio}."
                                )

                            graph_seq.append(data)

                        cache_object = {
                            "graph_seq": graph_seq,
                            "metadata": {
                                "case_id": case_directory.name,
                                "prefix": pref,
                                "chunk_start_gr": float(chunk_frame_keys[0][0]),
                                "chunk_end_gr": float(chunk_frame_keys[-1][0]),
                                "chunk_start_pr": float(chunk_frame_keys[0][1]),
                                "chunk_end_pr": float(chunk_frame_keys[-1][1]),
                                "num_sampled_frames": int(len(graph_seq)),
                                "phase_label_id": int(label),
                                "raw_phase_label": int(raw_label),
                                "feature_cleaning": {
                                    "case_id_zeroed": bool(REMOVE_CASE_ID_FEATURE),
                                    "g_ratio_zeroed": bool(REMOVE_G_RATIO_FEATURE),
                                    "p_ratio_zeroed": bool(REMOVE_P_RATIO_FEATURE),
                                },
                            },
                        }

                        torch.save(
                            cache_object,
                            temporary_dir / f"frag_{global_idx:06d}.pt",
                        )
                        case_fragment_counts[case_directory.name] += 1
                        global_idx += 1

            empty_cases = [
                case_id
                for case_id, fragment_count in case_fragment_counts.items()
                if fragment_count == 0
            ]
            if empty_cases:
                raise RuntimeError(
                    "No retained binary fragment was created for:\n"
                    + "\n".join(empty_cases)
                )
            if global_idx == 0:
                raise RuntimeError("No binary fragments were created.")

            manifest = {
                **self._expected_manifest_fields(),
                "num_fragments": int(global_idx),
                "case_fragment_counts": case_fragment_counts,
                "completed_at": datetime.now().isoformat(),
            }
            (temporary_dir / "cache_manifest.json").write_text(
                json.dumps(manifest, indent=2, ensure_ascii=False),
                encoding="utf-8",
            )

            if self.binary_dir.exists():
                self._safe_remove_directory(self.binary_dir, self.root_dir)
            temporary_dir.rename(self.binary_dir)

        except Exception:
            if temporary_dir.exists():
                self._safe_remove_directory(temporary_dir, self.root_dir)
            raise

        print(f"[OK] Complete binary cache created: {global_idx} fragments.")

    def len(self):
        return len(self.files)

    def _load_raw(self, idx):
        return torch.load(
            self.files[int(idx)],
            map_location="cpu",
            weights_only=False,
        )

    def get_with_meta(self, idx):
        obj = self._load_raw(idx)

        if not isinstance(obj, dict) or "graph_seq" not in obj:
            raise RuntimeError(
                f"Unsupported cache object in {self.files[int(idx)]}."
            )

        graph_seq = obj["graph_seq"]
        meta = dict(obj.get("metadata", {}) or {})
        meta["fragment_file"] = self.files[int(idx)].name

        if not graph_seq:
            raise RuntimeError(
                f"Empty graph sequence in {self.files[int(idx)]}."
            )

        return graph_seq, meta

    def get(self, idx):
        seq, _ = self.get_with_meta(idx)
        return seq

In [ ]:
# =========================================================
# 4) 14/3/3 CASE-WISE SPLIT
# =========================================================
def collect_fragment_metadata(dataset):
    rows = []

    for idx in range(len(dataset)):
        seq, meta = dataset.get_with_meta(idx)
        y = int(seq[0].y.item())
        case_id = str(meta.get("case_id", "UNKNOWN"))

        rows.append({
            "idx": int(idx),
            "case_id": case_id,
            "phase_id": int(y),
            "phase_name": PHASE_NAMES[y] if 0 <= y < len(PHASE_NAMES) else f"Phase_{y}",
            "fragment_file": str(meta.get("fragment_file", "")),
            "prefix": str(meta.get("prefix", "")),
        })

    return pd.DataFrame(rows)


def phase_counts_for_indices(meta_df, indices):
    sub = meta_df[meta_df["idx"].isin([int(x) for x in indices])]
    counts = np.zeros(NUM_PHASES, dtype=int)

    for y, c in sub["phase_id"].value_counts().items():
        counts[int(y)] = int(c)

    return counts


def split_from_cases(meta_df, train_cases, val_cases, test_cases):
    train_idx = meta_df[meta_df["case_id"].isin(train_cases)]["idx"].to_numpy(dtype=int)
    val_idx = meta_df[meta_df["case_id"].isin(val_cases)]["idx"].to_numpy(dtype=int)
    test_idx = meta_df[meta_df["case_id"].isin(test_cases)]["idx"].to_numpy(dtype=int)
    return train_idx, val_idx, test_idx


def l1_to_full_distribution(counts, full_dist):
    if counts.sum() == 0:
        return 999.0
    dist = counts / counts.sum()
    return float(np.abs(dist - full_dist).sum())


def score_case_split(meta_df, train_idx, val_idx, test_idx):
    all_idx = meta_df["idx"].to_numpy(dtype=int)

    full_counts = phase_counts_for_indices(meta_df, all_idx)
    full_dist = full_counts / max(full_counts.sum(), 1)

    counts_train = phase_counts_for_indices(meta_df, train_idx)
    counts_val = phase_counts_for_indices(meta_df, val_idx)
    counts_test = phase_counts_for_indices(meta_df, test_idx)

    coverage_train = int((counts_train > 0).sum())
    coverage_val = int((counts_val > 0).sum())
    coverage_test = int((counts_test > 0).sum())

    # Penalize empty classes in val/test, distribution mismatch, and severe small split sizes.
    score = (
        120.0 * coverage_test
        + 100.0 * coverage_val
        + 20.0 * coverage_train
        - 20.0 * l1_to_full_distribution(counts_test, full_dist)
        - 15.0 * l1_to_full_distribution(counts_val, full_dist)
        - 5.0 * l1_to_full_distribution(counts_train, full_dist)
    )

    # Extra penalty if val/test miss many classes.
    if coverage_test < min(NUM_PHASES, 6):
        score -= 200.0
    if coverage_val < min(NUM_PHASES, 6):
        score -= 150.0

    return score, {
        "coverage_train": coverage_train,
        "coverage_val": coverage_val,
        "coverage_test": coverage_test,
        "counts_train": counts_train,
        "counts_val": counts_val,
        "counts_test": counts_test,
        "full_counts": full_counts,
    }


def build_case_wise_14_3_3_split(meta_df):
    cases = sorted(meta_df["case_id"].unique().tolist())

    if "UNKNOWN" in cases:
        raise RuntimeError(
            "Case-wise split requires metadata case_id. Cache contains UNKNOWN case_id."
        )

    if len(cases) != EXPECTED_NUM_CASES:
        raise RuntimeError(
            f"This script expects exactly {EXPECTED_NUM_CASES} source-development cases for 14/3/3 split, "
            f"but found {len(cases)} cases: {cases}\n"
            "Either add/remove cases or adjust EXPECTED_NUM_CASES, TRAIN_CASES_N, VAL_CASES_N, TEST_CASES_N."
        )

    if TRAIN_CASES_N + VAL_CASES_N + TEST_CASES_N != len(cases):
        raise RuntimeError(
            f"Split counts do not sum to number of cases: "
            f"{TRAIN_CASES_N}+{VAL_CASES_N}+{TEST_CASES_N} != {len(cases)}"
        )

    best = None

    for trial in range(CASE_SPLIT_SEARCH_TRIALS):
        rng = np.random.default_rng(SEED + trial)
        shuffled = cases.copy()
        rng.shuffle(shuffled)

        train_cases = sorted(shuffled[:TRAIN_CASES_N])
        val_cases = sorted(shuffled[TRAIN_CASES_N:TRAIN_CASES_N + VAL_CASES_N])
        test_cases = sorted(shuffled[TRAIN_CASES_N + VAL_CASES_N:])

        train_idx, val_idx, test_idx = split_from_cases(meta_df, train_cases, val_cases, test_cases)

        if len(train_idx) == 0 or len(val_idx) == 0 or len(test_idx) == 0:
            continue

        score, diag = score_case_split(meta_df, train_idx, val_idx, test_idx)

        if best is None or score > best["score"]:
            best = {
                "score": float(score),
                "trial": int(trial),
                "train_cases": train_cases,
                "val_cases": val_cases,
                "test_cases": test_cases,
                "train_idx": train_idx,
                "val_idx": val_idx,
                "test_idx": test_idx,
                "diag": diag,
            }

    if best is None:
        raise RuntimeError("Could not build a valid 14/3/3 case-wise split.")

    split_info = {
        "split_mode": "automatic_case_wise_14_3_3_stratified_search_clean_caseid_zeroed",
        "num_cases": int(len(cases)),
        "train_cases_n": int(TRAIN_CASES_N),
        "val_cases_n": int(VAL_CASES_N),
        "test_cases_n": int(TEST_CASES_N),
        "search_trials": int(CASE_SPLIT_SEARCH_TRIALS),
        "best_trial": int(best["trial"]),
        "best_score": float(best["score"]),
        "train_cases": best["train_cases"],
        "val_cases": best["val_cases"],
        "test_cases": best["test_cases"],
        "coverage_train": int(best["diag"]["coverage_train"]),
        "coverage_val": int(best["diag"]["coverage_val"]),
        "coverage_test": int(best["diag"]["coverage_test"]),
        "feature_cleaning": {
            "case_id_zeroed": bool(REMOVE_CASE_ID_FEATURE),
            "g_ratio_zeroed": bool(REMOVE_G_RATIO_FEATURE),
            "p_ratio_zeroed": bool(REMOVE_P_RATIO_FEATURE),
        },
    }

    return best["train_idx"], best["val_idx"], best["test_idx"], split_info


def build_split(dataset, results_dir):
    meta_df = collect_fragment_metadata(dataset)

    train_idx, val_idx, test_idx, split_info = build_case_wise_14_3_3_split(meta_df)

    np.savez(
        os.path.join(results_dir, "data_split.npz"),
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
    )

    with open(os.path.join(results_dir, "split_cases.json"), "w", encoding="utf-8") as f:
        json.dump(split_info, f, indent=2, ensure_ascii=False)

    meta_df.to_csv(os.path.join(results_dir, "fragment_metadata.csv"), index=False)

    rows = []
    for split_name, idxs in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        counts = phase_counts_for_indices(meta_df, idxs)
        for pid, c in enumerate(counts):
            rows.append({
                "split": split_name,
                "phase_id": pid,
                "phase_name": PHASE_NAMES[pid],
                "num_fragments": int(c),
            })

    phase_counts_df = pd.DataFrame(rows)
    phase_counts_df.to_csv(os.path.join(results_dir, "split_phase_counts.csv"), index=False)

    split_summary_df = pd.DataFrame([
        {
            "split": "train",
            "num_fragments": int(len(train_idx)),
            "num_cases": int(len(split_info["train_cases"])),
            "cases": ", ".join(split_info["train_cases"]),
        },
        {
            "split": "val",
            "num_fragments": int(len(val_idx)),
            "num_cases": int(len(split_info["val_cases"])),
            "cases": ", ".join(split_info["val_cases"]),
        },
        {
            "split": "test",
            "num_fragments": int(len(test_idx)),
            "num_cases": int(len(split_info["test_cases"])),
            "cases": ", ".join(split_info["test_cases"]),
        },
    ])
    split_summary_df.to_csv(os.path.join(results_dir, "split_fragment_summary.csv"), index=False)

    print("\n[SPLIT SUMMARY]")
    print(split_summary_df)

    print("\n[SPLIT PHASE COUNTS]")
    print(phase_counts_df.pivot(index="phase_name", columns="split", values="num_fragments").fillna(0).astype(int))

    print("\n[AUTOMATIC 14/3/3 CASE-WISE SPLIT]")
    print(f"Train cases ({len(split_info['train_cases'])}): {split_info['train_cases']}")
    print(f"Val cases   ({len(split_info['val_cases'])})  : {split_info['val_cases']}")
    print(f"Test cases  ({len(split_info['test_cases'])}) : {split_info['test_cases']}")
    print(
        f"Phase coverage train/val/test: "
        f"{split_info['coverage_train']}/"
        f"{split_info['coverage_val']}/"
        f"{split_info['coverage_test']}"
    )

    return train_idx, val_idx, test_idx, split_info, meta_df

In [ ]:
# =========================================================
# 5) CLASS WEIGHTS
# =========================================================
def compute_auto_class_weights(dataset, indices, num_classes=8):
    print("[INFO] Computing class weights from train split...")

    counts = np.zeros(num_classes, dtype=np.float64)

    for idx in indices:
        sample = dataset.get(int(idx))
        label = int(sample[0].y.item())
        counts[label] += 1

    counts = np.clip(counts, a_min=1, a_max=None)
    weights = 1.0 / counts
    weights = weights / weights.sum() * num_classes

    print("[CLASS COUNTS]", counts.astype(int).tolist())
    print("[CLASS WEIGHTS]", [round(float(w), 4) for w in weights.tolist()])

    return torch.tensor(weights, dtype=torch.float)

In [ ]:
# =========================================================
# 6) MODEL
# =========================================================
class RobustMosaicGNN(nn.Module):
    def __init__(self, in_channels=11, edge_dim=3, hidden_dim=128, num_phases=8):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.conv1 = GATv2Conv(in_channels, hidden_dim, edge_dim=edge_dim)
        self.conv2 = GATv2Conv(hidden_dim, hidden_dim, edge_dim=edge_dim)

        self.context_attn = nn.Linear(hidden_dim * 2, 1)
        self.rnn = nn.GRU(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)

        self.fc = nn.Linear(hidden_dim * 3, num_phases)

    def forward(self, graph_seq):
        frame_embs = []

        for data in graph_seq:
            x = self.conv1(data.x, data.edge_index, data.edge_attr).relu()
            x = self.conv2(x, data.edge_index, data.edge_attr).relu()

            mean_emb = x.mean(dim=0, keepdim=True)

            v_node_emb = x[data.is_v]
            if v_node_emb.size(0) == 0:
                v_node_emb = mean_emb
            else:
                v_node_emb = v_node_emb.mean(dim=0, keepdim=True)

            combined = torch.cat([mean_emb, v_node_emb], dim=-1)
            attn_weight = torch.sigmoid(self.context_attn(combined))
            v = attn_weight * v_node_emb + (1.0 - attn_weight) * mean_emb

            frame_embs.append(v)

        seq_tensor = torch.stack(frame_embs).squeeze(1).unsqueeze(0)

        out, _ = self.rnn(seq_tensor)

        global_skip = seq_tensor.mean(dim=1)
        rnn_final = out[:, -1, :]

        final_combined = torch.cat([rnn_final, global_skip], dim=-1)

        return self.fc(final_combined)

In [ ]:
# =========================================================
# 7) EVAL HELPERS
# =========================================================
def calculate_accuracy(model, dataset, indices, device):
    model.eval()
    correct = 0

    with torch.no_grad():
        for idx in indices:
            seq = [d.to(device) for d in dataset.get(int(idx))]
            pred = model(seq).argmax(1).item()
            if pred == seq[0].y.item():
                correct += 1

    return (correct / max(len(indices), 1)) * 100.0


def write_ieee_style_report(
    file_path,
    checkpoint,
    train_acc,
    overall_acc,
    macro_p,
    macro_r,
    macro_f1,
    weighted_p,
    weighted_r,
    weighted_f1,
    y_true,
    y_pred,
    phase_names,
):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=range(len(phase_names)),
        zero_division=0,
    )

    with open(file_path, "w", encoding="utf-8") as f:
        f.write("=" * 90 + "\n")
        f.write("ROLE-NORMALIZED 14/3/3 CASE-WISE CLEAN S-STKG CLASSIFICATION REPORT (TEST SET)\n")
        f.write("=" * 90 + "\n\n")

        f.write("Model                : RobustMosaicGNN\n")
        f.write("Input semantic field : functional_role_id stored in node column 'class'\n")
        f.write("Split protocol       : automatic case-wise 14 train / 3 val / 3 test\n")
        f.write(f"Best Epoch           : {checkpoint['epoch']}\n")
        f.write(f"Train Accuracy       : {train_acc:.2f}%\n")
        f.write(f"Best Validation Acc  : {checkpoint['val_acc']:.2f}%\n\n")

        f.write("-" * 30 + " Overall Metrics " + "-" * 43 + "\n\n")
        f.write(f"Accuracy             : {overall_acc:.2f}%\n")
        f.write(f"Macro Precision      : {macro_p:.4f}\n")
        f.write(f"Macro Recall         : {macro_r:.4f}\n")
        f.write(f"Macro F1-score       : {macro_f1:.4f}\n")
        f.write(f"Weighted Precision   : {weighted_p:.4f}\n")
        f.write(f"Weighted Recall      : {weighted_r:.4f}\n")
        f.write(f"Weighted F1-score    : {weighted_f1:.4f}\n\n")

        f.write("-" * 90 + "\n")
        f.write("Per-Class Performance\n")
        f.write("-" * 90 + "\n\n")
        f.write(f"{'Class':<32}{'Precision':<12}{'Recall':<12}{'F1-Score':<12}{'Support':<10}\n")
        f.write("-" * 90 + "\n")

        for i, cls in enumerate(phase_names):
            f.write(f"{cls:<32}{precision[i]:<12.4f}{recall[i]:<12.4f}{f1[i]:<12.4f}{support[i]:<10}\n")

        f.write("\n" + "-" * 90 + "\n")
        f.write("Notes:\n")
        f.write("- Macro metrics treat all classes equally.\n")
        f.write("- Weighted metrics account for class imbalance.\n")
        f.write("- This run uses role-normalized node semantics, not raw YOLO class IDs.\n- Raw case_id is removed from input features by setting column 7 to 0.0.\n")
        f.write("- Train/validation/test splits are case-disjoint with automatic 14/3/3 case selection.\n")
        f.write("=" * 90 + "\n")

In [ ]:
# =========================================================
# 8) MAIN PIPELINE
# =========================================================
def run_phase_classifier_pipeline():
    set_seed(SEED)

    results_root = Path(RESULTS_ROOT).expanduser().resolve()
    results_root.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    results_dir = (
        results_root
        / f"source_sstkg_phase_classifier_{timestamp}"
    )
    results_dir.mkdir(parents=True, exist_ok=False)

    model_save_path = os.path.join(
        results_dir,
        "best_sstkg_phase_classifier.pt",
    )

    print("\n" + "=" * 90)
    print("[START] ROLE-NORMALIZED PHASE CLASSIFICATION — AUTO 14/3/3 CASE-WISE")
    print("=" * 90)
    print(f"[OUTPUT DIR] {results_dir}")
    print(f"[BASE PATH]  {BASE_PATH}")
    print(f"[BIN PATH]   {BIN_PATH}")
    print(f"[DEVICE]     {DEVICE}")
    print(f"[FEATURE CLEANING] case_id_zeroed={REMOVE_CASE_ID_FEATURE}, g_ratio_zeroed={REMOVE_G_RATIO_FEATURE}, p_ratio_zeroed={REMOVE_P_RATIO_FEATURE}")

    dataset = SurgicalTurboDataset(
        BASE_PATH,
        BIN_PATH,
        chunk_size=CHUNK_SIZE,
        minimum_chunk_frames=MIN_CHUNK_FRAMES,
    )

    train_idx, val_idx, test_idx, split_info, meta_df = build_split(dataset, results_dir)

    class_weights = compute_auto_class_weights(dataset, train_idx, num_classes=NUM_PHASES).to(DEVICE)

    model = RobustMosaicGNN(hidden_dim=HIDDEN_DIM, num_phases=NUM_PHASES).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
    )

    history = {
        "loss": [],
        "train_acc": [],
        "val_acc": [],
        "lr": [],
    }

    best_val_acc = float("-inf")

    print("\n[TRAINING STARTED]")

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        correct = 0
        start_t = time.time()

        for idx in train_idx:
            seq = [d.to(DEVICE) for d in dataset.get(int(idx))]

            optimizer.zero_grad()
            out = model(seq)
            target = seq[0].y.to(DEVICE)

            loss = criterion(out, target)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()

            total_loss += float(loss.item())

            if out.argmax(1).item() == int(target.item()):
                correct += 1

        model.eval()
        v_correct = 0

        with torch.no_grad():
            for idx in val_idx:
                seq = [d.to(DEVICE) for d in dataset.get(int(idx))]
                pred = model(seq).argmax(1).item()
                if pred == seq[0].y.item():
                    v_correct += 1

        t_acc = (correct / max(len(train_idx), 1)) * 100.0
        v_acc = (v_correct / max(len(val_idx), 1)) * 100.0
        avg_loss = total_loss / max(len(train_idx), 1)

        scheduler.step(v_acc)

        current_lr = optimizer.param_groups[0]["lr"]

        history["loss"].append(avg_loss)
        history["train_acc"].append(t_acc)
        history["val_acc"].append(v_acc)
        history["lr"].append(current_lr)

        print(
            f"Ep {epoch:03d} | "
            f"Loss: {avg_loss:.4f} | "
            f"T_Acc: {t_acc:.1f}% | "
            f"V_Acc: {v_acc:.1f}% | "
            f"LR: {current_lr:.6f} | "
            f"{time.time() - start_t:.1f}s"
        )

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_acc": v_acc,
                    "role_normalized": True,
                    "split_mode": "automatic_case_wise_14_3_3_clean_caseid_zeroed",
                    "split_info": split_info,
                    "base_path": str(Path(BASE_PATH).resolve()),
                    "binary_cache": str(Path(BIN_PATH).resolve()),
                    "split_path": os.path.join(results_dir, "data_split.npz"),
                    "phase_names": PHASE_NAMES,
                    "feature_cleaning": {
                        "case_id_zeroed": bool(REMOVE_CASE_ID_FEATURE),
                        "g_ratio_zeroed": bool(REMOVE_G_RATIO_FEATURE),
                        "p_ratio_zeroed": bool(REMOVE_P_RATIO_FEATURE),
                    },
                    "config": {
                        "chunk_size": CHUNK_SIZE,
                        "hidden_dim": HIDDEN_DIM,
                        "num_epochs": NUM_EPOCHS,
                        "lr": LR,
                        "weight_decay": WEIGHT_DECAY,
                        "seed": SEED,
                        "train_cases_n": TRAIN_CASES_N,
                        "val_cases_n": VAL_CASES_N,
                        "test_cases_n": TEST_CASES_N,
                    },
                },
                model_save_path,
            )
            print(f"   [SAVE] New best model: {v_acc:.2f}%")

    # =========================================================
    # TEST
    # =========================================================
    print("\n" + "=" * 90)
    print("[REPORTING STARTED]")
    print("=" * 90)

    checkpoint = torch.load(model_save_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    train_acc = calculate_accuracy(model, dataset, train_idx, DEVICE)

    y_true = []
    y_pred = []

    with torch.no_grad():
        for idx in test_idx:
            seq = [d.to(DEVICE) for d in dataset.get(int(idx))]
            out = model(seq)
            y_true.append(int(seq[0].y.item()))
            y_pred.append(int(out.argmax(1).item()))

    y_true_np = np.asarray(y_true, dtype=int)
    y_pred_np = np.asarray(y_pred, dtype=int)

    overall_acc = accuracy_score(y_true_np, y_pred_np) * 100.0

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true_np,
        y_pred_np,
        average="macro",
        zero_division=0,
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true_np,
        y_pred_np,
        average="weighted",
        zero_division=0,
    )

    report_path = os.path.join(results_dir, "S-STKG_ROLE_20CASE_14_3_3_CLEAN_Test_Report_IEEE_Springer.txt")

    write_ieee_style_report(
        file_path=report_path,
        checkpoint=checkpoint,
        train_acc=train_acc,
        overall_acc=overall_acc,
        macro_p=macro_precision,
        macro_r=macro_recall,
        macro_f1=macro_f1,
        weighted_p=weighted_precision,
        weighted_r=weighted_recall,
        weighted_f1=weighted_f1,
        y_true=y_true_np,
        y_pred=y_pred_np,
        phase_names=PHASE_NAMES,
    )

    config = {
        "role_normalized": True,
        "base_path": str(Path(BASE_PATH).resolve()),
        "binary_cache": str(Path(BIN_PATH).resolve()),
        "results_dir": str(results_dir),
        "best_model_path": model_save_path,
        "split_path": os.path.join(results_dir, "data_split.npz"),
        "split_info": split_info,
        "feature_cleaning": {
            "case_id_zeroed": bool(REMOVE_CASE_ID_FEATURE),
            "g_ratio_zeroed": bool(REMOVE_G_RATIO_FEATURE),
            "p_ratio_zeroed": bool(REMOVE_P_RATIO_FEATURE),
        },
        "num_fragments": int(len(dataset)),
        "num_train": int(len(train_idx)),
        "num_val": int(len(val_idx)),
        "num_test": int(len(test_idx)),
        "best_epoch": int(checkpoint["epoch"]),
        "best_val_acc": float(checkpoint["val_acc"]),
        "test_accuracy": float(overall_acc),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_precision),
        "weighted_recall": float(weighted_recall),
        "weighted_f1": float(weighted_f1),
        "phase_names": PHASE_NAMES,
    }

    with open(os.path.join(results_dir, "run_config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2, ensure_ascii=False)

    # Training history
    plt.figure(figsize=(15, 6))

    plt.subplot(1, 2, 1)
    plt.plot(history["loss"], linewidth=2)
    plt.title("Training Loss Evolution")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(history["train_acc"], label="Train Accuracy")
    plt.plot(history["val_acc"], label="Validation Accuracy")
    plt.title("Accuracy Evolution (%)")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy %")
    plt.ylim([0, 100])
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(
        os.path.join(results_dir, "S-STKG_ROLE_20CASE_14_3_3_CLEAN_Training_History.pdf"),
        dpi=400,
        bbox_inches="tight",
    )
    plt.close()

    # Confusion matrix
    cm = confusion_matrix(
        y_true_np,
        y_pred_np,
        labels=range(NUM_PHASES),
    )

    confusion_figure, confusion_axis = plt.subplots(
        figsize=(10, 8)
    )
    confusion_display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            f"P{i + 1}" for i in range(NUM_PHASES)
        ],
    )
    confusion_display.plot(
        ax=confusion_axis,
        values_format="d",
        colorbar=False,
    )
    confusion_axis.set_title(
        f"ROLE-Normalized 20-case 14/3/3 Confusion Matrix (Test Set)\n"
        f"Accuracy: {overall_acc:.2f}% | Macro F1: {macro_f1:.4f} | "
        f"Weighted F1: {weighted_f1:.4f}"
    )
    confusion_figure.tight_layout()
    confusion_figure.savefig(
        os.path.join(
            results_dir,
            "S-STKG_ROLE_20CASE_14_3_3_CLEAN_Confusion_Matrix.png",
        ),
        dpi=400,
        bbox_inches="tight",
    )
    plt.close(confusion_figure)

    print(f"\n[OK] Best model saved: {model_save_path}")
    print(f"[OK] Split saved     : {os.path.join(results_dir, 'data_split.npz')}")
    print(f"[OK] Report saved    : {report_path}")
    print(f"[OK] Results dir     : {results_dir}")
    print(f"[TEST] Accuracy={overall_acc:.2f}% | MacroF1={macro_f1:.4f} | WeightedF1={weighted_f1:.4f}")

    return {
        "results_directory": Path(results_dir),
        "checkpoint_path": Path(model_save_path),
        "split_path": Path(results_dir) / "data_split.npz",
        "split_cases_path": Path(results_dir) / "split_cases.json",
        "report_path": Path(report_path),
        "metrics": {
            "test_accuracy": float(overall_acc),
            "macro_f1": float(macro_f1),
            "weighted_f1": float(weighted_f1),
        },
    }



In [ ]:
run_output = run_phase_classifier_pipeline()

run_output
